<p><font size="6" color='grey'> <b>
Machine Learning
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
Supervised Learning - Decision Tree - Titanic
</b></font> </br></p>

---


# 0  | Install & Import
***

In [ ]:
# Install
!uv pip install --system -q dtreeviz

In [ ]:
# Daten & Strukturen
from pandas import read_excel, DataFrame  # Excel laden und Daten verwalten

# Datenvorverarbeitung
from sklearn.impute import SimpleImputer  # Fehlende Werte imputieren
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder  # Kategoriale Variablen encodieren
from sklearn.preprocessing import MinMaxScaler, StandardScaler  # Features skalieren

# Datenaufteilung
from sklearn.model_selection import train_test_split  # Train/Test-Split

# Modell & Modell-Export
from sklearn.tree import DecisionTreeClassifier, export_text, export_graphviz  # Klassifikationsmodell und Export

# Evaluation & Metriken
from sklearn.metrics import (
    accuracy_score,  # Genauigkeit
    confusion_matrix,  # Konfusionsmatrix
    ConfusionMatrixDisplay,  # Visualisierung der Matrix
    classification_report,  # Detaillierter Bericht
)

# Modellanalyse & Visualisierung
from yellowbrick.model_selection import validation_curve, learning_curve  # Lern- und Validierungskurven
import graphviz  # Graphbasierte Visualisierung
import dtreeviz  # Erweiterte Baumvisualisierung

# Visualisierung
import plotly.express as px  # Interaktive Plots
import plotly.subplots as sp  # Subplots erstellen

In [ ]:
# Warnung ausstellen
import warnings
warnings.filterwarnings("ignore")

# 1  | Understand
***


<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Aufgabe verstehen</br>
✅ Daten sammeln</br>
✅ Statistische Analyse (Min, Max, Mean, Korrelation, ...)</br>
✅ Datenvisualisierung (Streudiagramm, Box-Plot, ...)</br>
✅ Prepare Schritte festlegen</br>

<p><font color='black' size="5">
📒 Anwendungsfall
</font></p>

Dies ist der legendäre Titanic ML-Wettbewerb – die beste erste Herausforderung, um in ML-Modellierung einzutauchen.

Die Aufgabe ist einfach: Mit maschinellem Lernen lässt sich ein Modell erstellen, das vorhersagt, welche Passagiere den Schiffbruch der Titanic überlebt haben.

[Titanic Org](https://www.encyclopedia-titanica.org/)

[DataSet](https://www.openml.org/search?type=data&status=active&id=41265)

[Info](https://www.kaggle.com/competitions/titanic/data)


**Datenfelder:**   
+ Age: Alter
+ Fare: Ticketpreis
+ Sex: Geschlecht (0 = männlich, 1 = weiblich)
+ sibsp: Der Datensatz definiert Familienbeziehungen auf diese Weise ... Geschwister = Bruder, Schwester, Stiefbruder, Stiefschwester Ehepartner = Ehemann, Ehefrau (Geliebte und Verlobte wurden ignoriert)
+ parch: Der Datensatz definiert Familienbeziehungen auf diese Weise ... Elternteil = Mutter, Vater Kind = Tochter, Sohn, Stieftochter, Stiefsohn. Einige Kinder reisten nur mit einem Kindermädchen, daher ist für sie Parch=0
+ Pclass: Passagierklasse, 1.- 3. Klasse
+ Embarked: Hafen der Einschiffung

In [ ]:
df = read_excel(
    "https://raw.githubusercontent.com/ralf-42/ML_Intro/main/02_daten/05_tabellen/titanic.xlsx",
    usecols=["pclass", "survived", "sex", "age", "sibsp", "parch"],
)

In [ ]:
data = df.copy()
target = data.pop("survived")

<p><font color='black' size="5">
🔎 EDA (Exploratory Data Analysis) mit Pandas
</font></p>

In [ ]:
data.info()

In [ ]:
data.describe().T

In [ ]:
data.groupby("sex").count()

In [ ]:
target.value_counts()

In [ ]:
data[data.age.isnull()].describe().T

#  2 | Prepare

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Train-Test-Split durchführen</br>
✅ Nicht benötigte Features löschen</br>
✅ Datentyp ermitteln/ändern</br>
✅ Missing Values behandeln</br>
✅ Ausreißer behandeln</br>
✅ Kategorischer Features Kodieren</br>
✅ Numerischer Features skalieren</br>
✅ Feature-Engineering (neue Features schaffen)</br>
✅ Dimensionalität reduzieren</br>
✅ Resampling (Over-/Undersampling)</br>
✅ Pipeline erstellen/konfigurieren</br>


<font color='black' size="5">
Datentyp ermitteln
</font>

In [ ]:
all_col = data.columns
num_col = data.select_dtypes(include="number").columns
cat_col = data.select_dtypes(exclude="number").columns

<font color='black' size="5">
Train-Test-Set
</font>

In [ ]:
data_train, data_test, target_train, target_test = train_test_split(
    data, target, test_size=0.20, stratify=target, random_state=42)

<font color='black' size="5">
Missing Values - kategoriale Features
</font>

In [ ]:
imputer_cat = SimpleImputer(strategy="most_frequent")
imputer_cat.fit(data_train[cat_col])
data_train[cat_col] = imputer_cat.transform(data_train[cat_col])
data_test[cat_col] = imputer_cat.transform(data_test[cat_col])

<font color='black' size="5">
Kodierung
</font>

In [ ]:
encoder = OrdinalEncoder()
encoder.fit(data_train[cat_col])
data_train[cat_col] = encoder.transform(data_train[cat_col])
data_test[cat_col] = encoder.transform(data_test[cat_col])

<font color='black' size="5">
Missing Values - numerische Features
</font>

In [ ]:
imputer_num = SimpleImputer(strategy="mean")
imputer_num.fit(data_train[num_col])
data_train[num_col] = imputer_num.transform(data_train[num_col])
data_test[num_col] = imputer_num.transform(data_test[num_col])

<font color='black' size="5">
Skalierung
</font>

In [ ]:
scaler = StandardScaler()
scaler.fit(data_train[all_col])
data_train[all_col] = scaler.transform(data_train[all_col])
data_test[all_col] = scaler.transform(data_test[all_col])

# 3 | Modeling
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Modellauswahl treffen</br>
✅ Pipeline erweitern/konfigurieren</br>
✅ Training durchführen</br>
✅ Hyperparameter Tuning</br>
✅ Cross-Valdiation</br>
✅ Bootstrapping</br>
✅ Regularization</br>

<p><font color='black' size="5">🌳 Algorithmus: Entscheidungsbaum (Decision Tree – Klassifikation)</font></p>

Ein Entscheidungsbaum trifft Entscheidungen wie ein Ablaufdiagramm: Ausgehend von einer Wurzel stellt er Ja/Nein-Fragen zu den Merkmalen und verzweigt je nach Antwort – bis er eine Vorhersage (Blatt) erreicht.

**Analogie:** Beim Ratespiel 'Wer bin ich?' werden nacheinander Fragen gestellt – am Ende steht die Antwort. Genau so arbeitet ein Entscheidungsbaum.

**Wichtige Parameter:**

| Parameter | Bedeutung |
|-----------|----------|
| `max_depth` | Maximale Tiefe (begrenzt Overfitting) |
| `criterion` | Gütekriterium für Splits: `gini` oder `entropy` |
| `random_state` | Reproduzierbarkeit |

**Wann Validation Curve?**

**In der Praxis relevant wenn:**
- Verstanden werden soll, wie ein bestimmter Hyperparameter (z.B. `max_depth`) die Modellgüte beeinflusst
- Overfitting oder Underfitting diagnostiziert werden soll: die Kurve zeigt, ab welchem Parameterwert das Modell zu komplex wird
- Der optimale Wertebereich für einen Parameter vor dem eigentlichen Hyperparameter-Tuning eingegrenzt werden soll

**Nicht geeignet wenn:**
- Mehrere Parameter gleichzeitig optimiert werden sollen → dann Grid Search oder Randomized Search
- Ein schneller Überblick über das Modellverhalten gefragt ist → dann reicht ein einfacher Train-Test-Split

**Typischer Fehler:**
Die Validation Curve auf den Testdaten berechnen. Die Kurve verwendet immer nur Trainingsdaten (mit CV-Splits) — Testdaten bleiben bis zur finalen Evaluation unberührt.

In [ ]:
#@markdown   <p><font size="4" color='green'>  Entscheidungsbaum</font> </br></p>

import base64
from IPython.display import Image, display

diagram = """
flowchart TD
    ROOT[Wurzel: Weiblich?] -->|ja| A[Klasse 1 oder 2?]
    ROOT -->|nein| B[Alter < 16?]

    A -->|ja| C[Überlebensrate: 94.2%]
    A -->|nein| D[Überlebensrate: 50.0%]

    B -->|ja| E[Überlebensrate: 38.2%]
    B -->|nein| F[Überlebensrate: 16.5%]
"""

encoded = base64.urlsafe_b64encode(diagram.strip().encode()).decode()
display(Image(url=f"https://mermaid.ink/img/{encoded}", width=700))

<p><font color='black' size="5">
Modellauswahl & Training
</font></p>

In [ ]:
model = DecisionTreeClassifier(max_depth=3, random_state=42)

In [ ]:
model.fit(data_train, target_train)

# 4 | Evaluate
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Prognose (Train, Test) erstellen</br>
✅ Modellgüte prüfen</br>
✅ Residuenanalyse erstellen</br>
✅ Feature Importance/Selektion prüfen</br>
✅ Robustheitstest erstellen</br>
✅ Modellinterpretation erstellen</br>
✅ Sensitivitätsanalyse erstellen</br>
✅ Kommunikation (Key Takeaways)</br>

<p><font color='black' size="5">
Prognose
</font></p>

In [ ]:
target_train_pred = model.predict(data_train)
target_test_pred = model.predict(data_test)

<p><font color='black' size="5">
Confusion Matrix
</font></p>

In [ ]:
conf_matrix = confusion_matrix(target_test, target_test_pred)
display_labels_ = ["Not Survived", "Survived"]
disp = ConfusionMatrixDisplay(conf_matrix, display_labels=display_labels_)
disp.plot(cmap="Blues")

In [ ]:
print(classification_report(target_test, target_test_pred, target_names=display_labels_))

<p><font color='black' size="5">
Accuracy
</font></p>

In [ ]:
acc_train = accuracy_score(target_train, target_train_pred) * 100
print(f"Modell: {model} -- Train -- Accuracy: {acc_train:5.2f}")

In [ ]:
acc_test = accuracy_score(target_test, target_test_pred) * 100
print(f"Modell: {model} -- Test -- Accuracy: {acc_test:5.2f}%")

<p><font color='black' size="5">
Aufbau Analysewürfel
</font></p>

In [ ]:
# Übernahme der Testdaten
cube = data_test.copy()
cube.reset_index(inplace=True)

In [ ]:
# Übernahme Target real & predict
cube["real"] = DataFrame(target_test.values, columns=["real"])
cube["predict"] = DataFrame(target_test_pred, columns=["predict"])

<p><font color='black' size="5">
Visualisierung real vs predict
</font></p>

In [ ]:
# Histogramm
title_ = "Histogramm real vs predict"
fig = px.histogram(cube, x=["real", "predict"], nbins=2, title=title_)
fig.update_layout(barmode="group", bargap=0.1, width=600, height=600)
fig.show()

<p><font color='black' size="5">
Fehlerhafte Vorhersagen
</font></p>

In [ ]:
# real <> predict
cube[cube.real != cube.predict].describe().T

In [ ]:
cube[cube.real != cube.predict]

<p><font color='black' size="5">
Einzelne Vorhersage
</font></p>

In [ ]:
# 2 neue Datensätze werden zur Prognose an das Modell übergeben: Rose & Jack (Winslet/diCaprio)
new_data = {
    "age": [22, 23],
    "sex": [1, 0],
    "sibsp": [0, 0],
    "parch": [0, 0],
    "pclass": [1, 3]
}
new = DataFrame(new_data)

In [ ]:
# Vorhersage erstellen Rose & Jack
model.predict(new)

<p><font color='black' size="5">
Feature Importance
</font></p>

In [ ]:
title_ = "Feature Importance Titanic"
px.bar(
    x=model.feature_importances_, y=data.columns, title=title_, width=800, height=600
).update_yaxes(categoryorder="total ascending")

<p><font color='black' size="5">
Validation Curve
</font></p>

In [ ]:
from sklearn.model_selection import StratifiedKFold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
viz = validation_curve(
    model,
    data_train,
    target_train,
    param_name="max_depth",
    param_range=range(1, 7),
    cv=cv,
    scoring="accuracy",
)

# 5 | Deploy
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Modellexport und -speicherung</br>
✅ Abhängigkeiten und Umgebung</br>
✅ Sicherheit und Datenschutz</br>
✅ In die Produktion integrieren</br>
✅ Tests und Validierung</br>
✅ Dokumentation & Wartung</br>